In [1]:
import polars as pl
import numpy as np
import torch
import torch.nn as nn
import pickle

In [2]:
model_dir = '/kaggle/input/models/karlmarkuskonstabel/cf-irt-predictor/pytorch/default/1'
user_map_path = f'{model_dir}/user_map.pkl'
problem_map_path = f'{model_dir}/problem_map.pkl'

with open(user_map_path, 'rb') as file:
    user_map = pickle.load(file)

with open(problem_map_path, 'rb') as file:
    problem_map = pickle.load(file)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [4]:
import torch.nn.functional as F

class IRTModel(nn.Module):
    def __init__(self, num_users, num_items):
        super().__init__()
        self.theta = nn.Embedding(num_users, 1)  # user ability
        self.beta  = nn.Embedding(num_items, 1)  # item difficulty
        self.alpha = nn.Embedding(num_items, 1)  # item discrimination

        nn.init.normal_(self.theta.weight, std=0.1)
        nn.init.normal_(self.beta.weight,  std=0.1)
        nn.init.ones_(self.alpha.weight)

    def forward(self, u, i):
        theta = self.theta(u).squeeze(-1)
        beta  = self.beta(i).squeeze(-1)
        alpha = F.softplus(self.alpha(i)).squeeze(-1)
        return alpha * (theta - beta)

In [5]:
model = IRTModel(num_users=len(user_map), num_items=len(problem_map)).to(device)
model.load_state_dict(torch.load(f"{model_dir}/irt_model.pt", weights_only=True))
model.eval()

IRTModel(
  (theta): Embedding(1722977, 1)
  (beta): Embedding(12316, 1)
  (alpha): Embedding(12316, 1)
)

In [6]:
def predict_prob(handle, problem_id):
    pred = model(torch.tensor([user_map[handle]], device=device), 
              torch.tensor([problem_map[problem_id]], device=device))
    return torch.sigmoid(pred).cpu().item()

In [7]:
p = predict_prob('TomatoPotato', '2217.H')
print(f"Predicted probability: {p:.4f}")

Predicted probability: 0.1143


In [8]:
import sys
sys.path.append('/kaggle/input/datasets/karlmarkuskonstabel/codeforces-interactions')

In [9]:
import cf_api

api = cf_api.CodeForcesAPI(None, None) # auth not needed
handled_verdicts = {
    "FAILED",
    "OK",
    "COMPILATION_ERROR",
    "RUNTIME_ERROR",
    "WRONG_ANSWER",
    "TIME_LIMIT_EXCEEDED",
    "MEMORY_LIMIT_EXCEEDED",
    "IDLENESS_LIMIT_EXCEEDED",
    "SECURITY_VIOLATED",
    "REJECTED",
}
def fetch_user_interactions(handle):
    submissions = api.request('user.status', {"handle": handle}, timeout=10)
    interactions = {}
    for submission in submissions:
        problem = submission['problem']
        if 'contestId' not in problem:
            continue
        problem_id = f'{problem["contestId"]}.{problem["index"]}'
        if problem_id not in problem_map:
            continue
        if submission['verdict'] not in handled_verdicts:
            continue
        interactions[problem_id] = interactions.get(problem_id, 0) or 1
    indices = []
    labels = []
    for problem_id, solved in interactions.items():
        indices.append(problem_map[problem_id])
        labels.append(float(solved))
    return torch.tensor(indices), torch.tensor(labels)

def estimate_new_user_theta(handle):
    problem_indices, labels = fetch_user_interactions(handle)
    num_problems = len(problem_indices)
    if num_problems == 0:
        return torch.tensor([0.0], device=device)
    problem_indices = problem_indices.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        solved_mask = labels == 1
        if solved_mask.any():
            initial_guess = model.beta(problem_indices[solved_mask]).mean()
        else:
            initial_guess = torch.tensor(0.0, device=device)

    new_theta = nn.Parameter(initial_guess.clone().detach())
    
    steps = max(50, 100 * int(np.log(num_problems + 1)))
    lr = 0.1
    if num_problems < 20:
        l2_lambda = 0.5
    elif num_problems < 200:
        l2_lambda = 0.1
    else:
        l2_lambda = 1.0 / num_problems
    optimizer = torch.optim.Adam([new_theta], lr=lr)
    pos_weight = torch.tensor([0.3], device=device)

    model.eval()
    for step in range(steps):
        beta  = model.beta(problem_indices).squeeze(-1)
        alpha = F.softplus(model.alpha(problem_indices)).squeeze(-1)
        preds = alpha * (new_theta - beta)
        loss = F.binary_cross_entropy_with_logits(preds, labels, pos_weight=pos_weight) + l2_lambda*((new_theta-initial_guess)**2)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return new_theta.detach()
    

In [10]:
def get_top_closest_to_difficulty(handle, target_prob, k=5):
    target_logit = torch.tensor(target_prob).logit()
    idx2problem = {v:k for k,v in problem_map.items()}

    num_problems = len(problem_map)
    solved_indices = set(idx.item() for idx, solved in zip(*fetch_user_interactions(handle)) if solved)
    unsolved_indices = set(range(num_problems)) - solved_indices
    unsolved_indices = torch.tensor(list(unsolved_indices), dtype=torch.long).to(device)
    if handle in user_map:
        theta = model.theta(torch.tensor([user_map[handle]], device=device, dtype=torch.long)).squeeze(-1)
    else:
        theta = estimate_new_user_theta(handle).expand(unsolved_indices.size(0)).to(device)
    
    with torch.no_grad():
        alpha = F.softplus(model.alpha(unsolved_indices)).squeeze(-1)
        beta = model.beta(unsolved_indices).squeeze(-1)
        logits = alpha * (theta - beta)
        distances = torch.abs(logits - target_logit)
        values, indices = torch.topk(distances, k, largest=False)
        return [(idx2problem[idx.cpu().item()], torch.sigmoid(logits[idx]).cpu().item()) for idx in indices]

In [12]:
handle = 'TomatoPotato'
theta = model.theta(torch.tensor(user_map[handle], device=device)).cpu().item()
estimation = estimate_new_user_theta(handle).item()
print(theta-estimation, theta, estimation)
recommended = get_top_closest_to_difficulty(handle, 0.4)
for problem_id, prob in recommended:
    contest_id, problem = problem_id.split('.') 
    print(f'https://codeforces.com/problemset/problem/{contest_id}/{problem} : {prob:.4f}')

0.37173140048980713 -0.6911898851394653 -1.0629212856292725
https://codeforces.com/problemset/problem/1516/D : 0.7001
https://codeforces.com/problemset/problem/922/D : 0.7001
https://codeforces.com/problemset/problem/1084/D : 0.6998
https://codeforces.com/problemset/problem/2152/H2 : 0.7002
https://codeforces.com/problemset/problem/1914/F : 0.7003
